# Build Prompt Data

Notebook nay dung de build du lieu prompt tu folder chua cac file parquet truoc khi chuyen sang `CustomTranslationDataset`.

In [ ]:
!hf auth login --token $HF_TOKEN

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `kaggle` has been saved to C:\Users\Asus\.cache\huggingface\stored_tokens
Your token has been saved to C:\Users\Asus\.cache\huggingface\token
Login successful.
The current active token is: `kaggle`


In [13]:
from pathlib import Path

import pandas as pd

from build_data import build_prompt_dataframe_from_folder
from typing import Optional, List, Dict, Any
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

In [6]:
folder_path = Path("test_data")
folder_path

WindowsPath('test_data')

## Direct Prompt Data

In [7]:
df_direct = build_prompt_dataframe_from_folder(
    folder_path=str(folder_path),
    prompt_type="direct",
)

print(df_direct.shape)
df_direct[["src_lang", "trg_lang", "src_text", "out_token_str", "instruction"]].head(3)

(100, 15)


,src_lang,trg_lang,src_text,out_token_str,instruction
0,en,vi,"""We now have 4-month-old mice that are non-dia...",“Chúng tôi hiện có đàn chuột 4 tháng tuổi đã h...,Translate the following text from English into...
1,en,vi,"Dr. Ehud Ur, professor of medicine at Dalhousi...","Tiến sĩ Ehud Ur, giáo sư y khoa của Trường Đại...",Translate the following text from English into...
2,en,vi,"Like some other experts, he is skeptical about...","Giống như một số chuyên gia khác, ông ấy hoài ...",Translate the following text from English into...


In [8]:
df_direct["instruction"][0]

'Translate the following text from English into Vietnamese:\nEnglish: "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.'

## Multipivot Prompt Data

In [9]:
df_multipivot = build_prompt_dataframe_from_folder(
    folder_path=str(folder_path),
    prompt_type="multipivot",
)

print(df_multipivot.shape)
df_multipivot[["src_lang", "trg_lang", "src_text", "out_token_str", "instruction"]].head(3)

(100, 15)


,src_lang,trg_lang,src_text,out_token_str,instruction
0,en,vi,"""We now have 4-month-old mice that are non-dia...",“Chúng tôi hiện có đàn chuột 4 tháng tuổi đã h...,Your task is to translate the SOURCE sentence ...
1,en,vi,"Dr. Ehud Ur, professor of medicine at Dalhousi...","Tiến sĩ Ehud Ur, giáo sư y khoa của Trường Đại...",Your task is to translate the SOURCE sentence ...
2,en,vi,"Like some other experts, he is skeptical about...","Giống như một số chuyên gia khác, ông ấy hoài ...",Your task is to translate the SOURCE sentence ...


## Compare Built Fields

In [10]:
common_cols = [
    "src_lang",
    "trg_lang",
    "src_text",
    "out_token_str",
    "instruction",
]

pd.concat(
    [
        df_direct[common_cols].assign(prompt_style="direct").head(2),
        df_multipivot[common_cols].assign(prompt_style="multipivot").head(2),
    ],
    ignore_index=True,
)

,src_lang,trg_lang,src_text,out_token_str,instruction,prompt_style
0,en,vi,"""We now have 4-month-old mice that are non-dia...",“Chúng tôi hiện có đàn chuột 4 tháng tuổi đã h...,Translate the following text from English into...,direct
1,en,vi,"Dr. Ehud Ur, professor of medicine at Dalhousi...","Tiến sĩ Ehud Ur, giáo sư y khoa của Trường Đại...",Translate the following text from English into...,direct
2,en,vi,"""We now have 4-month-old mice that are non-dia...",“Chúng tôi hiện có đàn chuột 4 tháng tuổi đã h...,Your task is to translate the SOURCE sentence ...,multipivot
3,en,vi,"Dr. Ehud Ur, professor of medicine at Dalhousi...","Tiến sĩ Ehud Ur, giáo sư y khoa của Trường Đại...",Your task is to translate the SOURCE sentence ...,multipivot


## Next Step

Neu muon chuyen sang `CustomTranslationDataset`, can build lai records voi `tokenizer=...` de co them `prefix_prompt` va `prompt`.

In [3]:

from transformers import AutoTokenizer
from build_data import build_prompt_records_from_folder
from custom_translation_dataset import CustomTranslationDataset

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
records_direct = build_prompt_records_from_folder(
    folder_path=str("test_data"),
    prompt_type="direct",
    tokenizer=tokenizer,
)
D_direct = CustomTranslationDataset(records_direct, tokenizer=tokenizer)

In [5]:
D_direct[0]

{'prompt_type': 'direct',
 'src_lang': 'en',
 'trg_lang': 'vi',
 'src_text': '"We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.',
 'out_token_str': '“Chúng tôi hiện có đàn chuột 4 tháng tuổi đã hết bị tiểu đường,” ông nói thêm.',
 'instruction': 'Translate the following text from English into Vietnamese:\nEnglish: "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.',
 'messages_prefix': [{'role': 'system',
   'content': 'You are a professional translator. Output ONLY the translation. Do not explain. Do not add notes. Do not add extra text.'},
  {'role': 'user',
   'content': 'Translate the following text from English into Vietnamese:\nEnglish: "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.'}],
 'messages_full': [{'role': 'system',
   'content': 'You are a professional translator. Output ONLY the translation. Do not explain. Do not add notes. Do not add extra text.'},
  

In [6]:
records_multipivot = build_prompt_records_from_folder(
    folder_path=str("test_data"),
    prompt_type="multipivot",
    tokenizer=tokenizer,
)

D_multipivot = CustomTranslationDataset(
    samples=records_multipivot,
    tokenizer=tokenizer,
)


In [7]:
assert len(D_direct) == len(D_multipivot)

for i in range(3):
    print(D_direct.preprocess_df_trans[i]["src_text"])
    print(D_direct.preprocess_df_trans[i]["out_token_str"])
    print(D_multipivot.preprocess_df_trans[i]["src_text"])
    print(D_multipivot.preprocess_df_trans[i]["out_token_str"])
    print("-" * 50)


"We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.
“Chúng tôi hiện có đàn chuột 4 tháng tuổi đã hết bị tiểu đường,” ông nói thêm.
"We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.
“Chúng tôi hiện có đàn chuột 4 tháng tuổi đã hết bị tiểu đường,” ông nói thêm.
--------------------------------------------------
Dr. Ehud Ur, professor of medicine at Dalhousie University in Halifax, Nova Scotia and chair of the clinical and scientific division of the Canadian Diabetes Association cautioned that the research is still in its early days.
Tiến sĩ Ehud Ur, giáo sư y khoa của Trường Đại học Dalhousie ở Halifax, Nova Scotia và là chủ tịch ban lâm sàng và khoa học của Hiệp hội Bệnh Tiểu đường Canada cảnh báo rằng nghiên cứu này chỉ mới là sự khởi đầu.
Dr. Ehud Ur, professor of medicine at Dalhousie University in Halifax, Nova Scotia and chair of the clinical and scientific division of the Canadian Diabetes Association caut

In [ ]:
def _load_model(model_name: str, device: Optional[str] = None):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

  
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )

    if device != "cuda":
        model = model.to(device)

    model.eval()
    return model, tokenizer, device
model, tokenizer, device = _load_model("meta-llama/Meta-Llama-3-8B-Instruct")

`torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\Asus\Desktop\exploring_translation_mechanism\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--meta-llama--Meta-Llama-3-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 4 files:   0%|  

## Baseline PPL

Tinh perplexity tren toan bo target span cho hai prompt style.

In [ ]:
from translation_perplexity_utils import (
    get_generated_token_perplexity,
    scan_path_patching_by_generated_perplexity,
)

ppl_direct = get_generated_token_perplexity(model, D_direct)
ppl_multipivot = get_generated_token_perplexity(model, D_multipivot)

print('Direct PPL:', float(ppl_direct))
print('Multipivot PPL:', float(ppl_multipivot))


## Scan Important Components

Chieu 1: patch activation tu multipivot sang direct va do thay doi PPL.

In [ ]:
baseline_ppl_direct, head_results_mp_to_direct, mlp_results_mp_to_direct = scan_path_patching_by_generated_perplexity(
    model,
    translation_dataset=D_direct,
    flipped_translation_dataset=D_multipivot,
    receiver_hooks=[(f'blocks.{model.cfg.n_layers - 1}.hook_resid_post', None)],
    batch_size=16,
)

print('Baseline direct PPL:', baseline_ppl_direct)
print('Head results shape:', tuple(head_results_mp_to_direct.shape))
print('MLP results shape:', tuple(mlp_results_mp_to_direct.shape))


## Reverse Direction

Chieu 2: patch activation tu direct sang multipivot.

In [ ]:
baseline_ppl_multipivot, head_results_direct_to_mp, mlp_results_direct_to_mp = scan_path_patching_by_generated_perplexity(
    model,
    translation_dataset=D_multipivot,
    flipped_translation_dataset=D_direct,
    receiver_hooks=[(f'blocks.{model.cfg.n_layers - 1}.hook_resid_post', None)],
    batch_size=16,
)

print('Baseline multipivot PPL:', baseline_ppl_multipivot)
print('Head results shape:', tuple(head_results_direct_to_mp.shape))
print('MLP results shape:', tuple(mlp_results_direct_to_mp.shape))


## Inspect Top Heads

Lay nhanh cac head co anh huong lon nhat trong chieu multipivot -> direct.

In [ ]:
top_k = 10
flat_vals, flat_idx = head_results_mp_to_direct.flatten().topk(top_k)
top_heads = []
for value, idx in zip(flat_vals.tolist(), flat_idx.tolist()):
    layer = idx // model.cfg.n_heads
    head = idx % model.cfg.n_heads
    top_heads.append({'layer': layer, 'head': head, 'delta_pct': value})

top_heads
